In [0]:

# ============================================================
# DIAGNÓSTICO: NOMBRES REALES DE LAS COLUMNAS BTS
# ============================================================

# Consultamos directamente el esquema de Bronze.
# No modificamos ni descargamos ningún dato.

tabla = "workspace.bronze_flights.bts_flights_raw"

columnas = spark.table(tabla).columns

print("Total de columnas:", len(columnas))

print("\nPRIMERAS 30 COLUMNAS:")
for columna in columnas[:30]:
    print(repr(columna))

print("\nCOLUMNAS RELACIONADAS CON VUELOS Y RETRASOS:")
for columna in columnas:
    if any(
        palabra in columna.upper()
        for palabra in [
            "FLIGHT", "ORIGIN", "DEST",
            "DEP", "CANCEL", "AIRLINE"
        ]
    ):
        print(repr(columna))


In [0]:

# ============================================================
# CELDA 1. PREPARACIÓN DE DATOS PARA SILVER
# ============================================================

# Recuperamos los vuelos originales de Bronze y
# normalizamos los nombres de las columnas.
#
# No modificamos la tabla Bronze.
# Trabajamos con febrero de 2024 a diciembre de 2025
# y los 30 aeropuertos seleccionados por el equipo.

from pyspark.sql import functions as F

TABLA_BRONZE = "workspace.bronze_flights.bts_flights_raw"
TABLA_AEROPUERTOS = "workspace.bronze_flights.weather_airports_top30"
TABLA_SILVER = "workspace.silver_flights.flights_clean"

# ------------------------------------------------------------
# 1. LEER Y NORMALIZAR LAS COLUMNAS
# ------------------------------------------------------------

df_bronze = spark.table(TABLA_BRONZE)

# Cambiamos los nombres a mayúsculas en nuestro DataFrame.
# Bronze conserva sus nombres originales.
df_bronze = df_bronze.toDF(
    *[columna.strip().upper() for columna in df_bronze.columns]
)

# Revisamos las columnas necesarias para la limpieza.
columnas_necesarias = [
    "FLIGHTDATE",
    "ORIGIN",
    "DEST",
    "CRSDEPTIME",
    "DEPDELAY",
    "CANCELLED",
    "REPORTING_AIRLINE"
]

faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in df_bronze.columns
]

if faltantes:
    raise ValueError(
        f"Faltan columnas BTS necesarias: {faltantes}"
    )

# ------------------------------------------------------------
# 2. FILTRAR EL PERIODO
# ------------------------------------------------------------

df_periodo = df_bronze.filter(
    (F.col("_SOURCE_YEAR") == 2025) |
    (
        (F.col("_SOURCE_YEAR") == 2024) &
        (F.col("_SOURCE_MONTH") >= 2)
    )
)

# ------------------------------------------------------------
# 3. SELECCIONAR LOS 30 AEROPUERTOS
# ------------------------------------------------------------

df_top30 = (
    spark.table(TABLA_AEROPUERTOS)
    .select(F.col("airport").alias("ORIGIN"))
    .distinct()
)

df_vuelos_silver_base = df_periodo.join(
    F.broadcast(df_top30),
    on="ORIGIN",
    how="inner"
)

# ------------------------------------------------------------
# 4. COMPROBAR LA PREPARACIÓN
# ------------------------------------------------------------

print("=" * 50)
print("PREPARACIÓN DE SILVER - VUELOS")
print("=" * 50)

print("Aeropuertos seleccionados:", df_top30.count())
print("Columnas disponibles:", len(df_vuelos_silver_base.columns))

display(
    df_vuelos_silver_base.select(
        "FLIGHTDATE",
        "ORIGIN",
        "DEST",
        "CRSDEPTIME",
        "DEPDELAY",
        "CANCELLED",
        "REPORTING_AIRLINE"
    ).limit(10)
)

print("PREPARACIÓN EXITOSA: datos listos para limpieza.")


In [0]:

# ============================================================
# CELDA 2. LIMPIEZA DE VUELOS Y CREACIÓN DE SILVER
# ============================================================

# Objetivo:
# Limpiar los vuelos de BTS y preparar los datos
# para predecir retrasos de salida de 15 minutos o más.
#
# Utilizamos únicamente los vuelos de los 30 aeropuertos
# seleccionados, entre febrero de 2024 y diciembre de 2025.
#
# Bronze permanece intacto.

from pyspark.sql import functions as F

TABLA_SILVER = "workspace.silver_flights.flights_clean"

# ------------------------------------------------------------
# 1. SELECCIONAR Y CONVERTIR LAS COLUMNAS
# ------------------------------------------------------------

df_limpio = (
    df_vuelos_silver_base
    .select(
        F.to_date("FLIGHTDATE").alias("flight_date"),
        F.col("ORIGIN").alias("origin"),
        F.col("DEST").alias("dest"),
        F.col("REPORTING_AIRLINE").alias("airline"),
        F.col("TAIL_NUMBER").alias("tail_number"),
        F.col("CRSDEPTIME").cast("int").alias("crs_dep_time"),
        F.col("DEPDELAY").cast("double").alias("dep_delay"),
        F.col("CANCELLED").cast("double").alias("cancelled"),
        F.col("_SOURCE_YEAR").alias("source_year"),
        F.col("_SOURCE_MONTH").alias("source_month")
    )
)

# ------------------------------------------------------------
# 2. EXCLUIR CANCELACIONES Y DATOS INCOMPLETOS
# ------------------------------------------------------------

# Los vuelos cancelados no tienen un retraso de salida
# comparable y no pueden recibir nuestra etiqueta.
df_limpio = df_limpio.filter(
    (F.col("cancelled") == 0) &
    F.col("dep_delay").isNotNull() &
    F.col("flight_date").isNotNull() &
    F.col("origin").isNotNull() &
    F.col("dest").isNotNull() &
    F.col("airline").isNotNull() &
    F.col("crs_dep_time").between(0, 2359)
)

# Validamos que los minutos programados sean correctos.
df_limpio = df_limpio.filter(
    (F.col("crs_dep_time") % 100) < 60
)

# ------------------------------------------------------------
# 3. CREAR VARIABLES TEMPORALES
# ------------------------------------------------------------

# Calculamos la hora programada de salida.
# CRSDEPTIME está almacenado como HHMM.
df_limpio = (
    df_limpio
    .withColumn(
        "departure_hour",
        F.floor(F.col("crs_dep_time") / 100).cast("int")
    )
    .withColumn(
        "departure_minute",
        (F.col("crs_dep_time") % 100).cast("int")
    )
    .withColumn("year", F.year("flight_date"))
    .withColumn("month", F.month("flight_date"))
    .withColumn(
        "day_of_week",
        F.dayofweek("flight_date")
    )
)

# ------------------------------------------------------------
# 4. CREAR LA VARIABLE OBJETIVO
# ------------------------------------------------------------

# 1: vuelo salió con 15 minutos o más de retraso.
# 0: vuelo salió con menos de 15 minutos de retraso.
df_limpio = df_limpio.withColumn(
    "delayed_15",
    F.when(F.col("dep_delay") >= 15, 1).otherwise(0)
)

# ------------------------------------------------------------
# 5. GUARDAR EN DELTA LAKE
# ------------------------------------------------------------

# Silver es una tabla derivada de Bronze.
# Sobrescribimos solamente la tabla Silver para que
# la ejecución sea reproducible y no duplique vuelos.
(
    df_limpio.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_SILVER)
)

# ------------------------------------------------------------
# 6. VALIDAR LA TABLA GENERADA
# ------------------------------------------------------------

df_silver = spark.table(TABLA_SILVER)

print("=" * 50)
print("RESULTADO DE SILVER - VUELOS")
print("=" * 50)

print("Registros limpios:", f"{df_silver.count():,}")
print(
    "Aeropuertos:",
    df_silver.select("origin").distinct().count()
)

print(
    "Tasa de vuelos retrasados:",
    f"{df_silver.agg(F.avg('delayed_15')).first()[0] * 100:.2f}%"
)

display(
    df_silver.groupBy("year", "month")
    .count()
    .orderBy("year", "month")
)

print("SILVER DE VUELOS CREADO CORRECTAMENTE.")


In [0]:

# ============================================================
# CELDA 3. VALIDACIÓN DE SILVER DE VUELOS
# ============================================================

# Comprobamos que la tabla limpia no tenga valores
# faltantes en las columnas esenciales del modelo.
#
# También revisamos que nuestra variable objetivo
# contenga únicamente valores 0 y 1.

from pyspark.sql import functions as F

TABLA_SILVER = "workspace.silver_flights.flights_clean"

df = spark.table(TABLA_SILVER)

# Columnas esenciales para los modelos.
columnas = [
    "flight_date",
    "origin",
    "dest",
    "airline",
    "crs_dep_time",
    "dep_delay",
    "delayed_15"
]

# Contamos nulos en una sola operación.
nulos = df.agg(*[
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in columnas
]).first().asDict()

print("NULOS EN VARIABLES ESENCIALES")

for columna, total in nulos.items():
    print(f"{columna}: {total:,}")

# Revisamos la distribución de la variable objetivo.
print("\nDISTRIBUCIÓN DE DELAYED_15")

display(
    df.groupBy("delayed_15")
    .count()
    .orderBy("delayed_15")
)

# Comprobamos el rango temporal y los aeropuertos.
resumen = df.agg(
    F.min("flight_date").alias("primera_fecha"),
    F.max("flight_date").alias("ultima_fecha"),
    F.countDistinct("origin").alias("aeropuertos")
).first()

print("\nCOBERTURA")
print("Primera fecha:", resumen["primera_fecha"])
print("Última fecha:", resumen["ultima_fecha"])
print("Aeropuertos:", resumen["aeropuertos"])

# Validación final.
if (
    all(v == 0 for v in nulos.values())
    and set(
        fila["delayed_15"]
        for fila in df.select("delayed_15").distinct().collect()
    ) == {0, 1}
    and resumen["aeropuertos"] == 30
):
    print("\nVALIDACIÓN EXITOSA: Silver listo.")
else:
    print("\nREVISAR: se encontraron inconsistencias.")
